In [1]:
import yfinance as yf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
# let's download 5 years of Apple stock data
df= yf.download("AAPL" , start ='2019-01-01', end ='2024-01-01')
print(df.shape)
print(df.head())

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

(1258, 5)
Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2019-01-02  37.503719  37.724583  36.627397  36.784138  148158800
2019-01-03  33.768070  34.606394  33.722947  34.193168  365248800
2019-01-04  35.209621  35.278494  34.150437  34.323801  234428400
2019-01-07  35.131245  35.344984  34.649149  35.314110  219111200
2019-01-08  35.800949  36.055060  35.271357  35.518341  164101200


In [2]:
#drops any missing data
print(df.isnull().sum())
df.dropna(inplace=True)

Price   Ticker
Close   AAPL      0
High    AAPL      0
Low     AAPL      0
Open    AAPL      0
Volume  AAPL      0
dtype: int64


In [3]:
df_2022 =df["2022-01-01":"20212-12-31"]
df_range =df["2021-06-01":"2021-12-31"]
print(df.index)
print(df.index.year)
print(df.index.dayofweek)

DatetimeIndex(['2019-01-02', '2019-01-03', '2019-01-04', '2019-01-07',
               '2019-01-08', '2019-01-09', '2019-01-10', '2019-01-11',
               '2019-01-14', '2019-01-15',
               ...
               '2023-12-15', '2023-12-18', '2023-12-19', '2023-12-20',
               '2023-12-21', '2023-12-22', '2023-12-26', '2023-12-27',
               '2023-12-28', '2023-12-29'],
              dtype='datetime64[ns]', name='Date', length=1258, freq=None)
Index([2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019,
       ...
       2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023],
      dtype='int32', name='Date', length=1258)
Index([2, 3, 4, 0, 1, 2, 3, 4, 0, 1,
       ...
       4, 0, 1, 2, 3, 4, 1, 2, 3, 4],
      dtype='int32', name='Date', length=1258)


In [4]:
# Daily return : how much did the price change from yesterday?
df["daily_return"] = df["Close"].pct_change()
print(df['daily_return'].head(10))

Date
2019-01-02         NaN
2019-01-03   -0.099607
2019-01-04    0.042690
2019-01-07   -0.002226
2019-01-08    0.019063
2019-01-09    0.016982
2019-01-10    0.003197
2019-01-11   -0.009818
2019-01-14   -0.015037
2019-01-15    0.020467
Name: daily_return, dtype: float64


In [5]:
def compute_rsi(series, window =14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = delta.clip(upper=0)

    #average gain and losses over 14 days
    avg_gain = gain.rolling(window=window).mean()
    avg_loss = -loss.rolling(window=window).mean()

    #formula
    rs = avg_gain/avg_loss
    rsi = 100 -(100/(1 + rs))

    return(rsi)

In [6]:
#rolling averages (7 -21 - 50 days) for closing
df['SMA_7'] = df["Close"].rolling(window =7).mean()
df['SMA_21'] =df["Close"].rolling(window=21).mean()
df['SMA_50'] = df["Close"].rolling(window=50).mean()

# n-day rolling standard deviation (volatility proxy)
df["volatility_7"] = df['Close'].rolling(window=7).std()
df['volatility_21'] = df['Close'].rolling(window= 21).std()
df['volatility_50'] = df['Close'].rolling(window=50).std()

#bollinger bands
df['BB_upper'] = df['SMA_21'] + 2 * df['volatility_21']
df['BB_lower'] = df['SMA_21'] - 2 * df['volatility_21']

#Relative Strength index flag
df['RSI_14'] = compute_rsi(df['Close'])

# lagged features
df['return_lag1'] = df['daily_return'].shift(1)
df['return_lag5'] = df['daily_return'].shift(5)
df['return_lag10'] = df['daily_return'].shift(10)

#clean up after any missing values that were just computed
df.dropna(inplace=True)
print(df[['volatility_7','volatility_21','volatility_50','RSI_14','BB_upper','BB_lower','return_lag1','return_lag5','return_lag10']].head())

Price      volatility_7 volatility_21 volatility_50     RSI_14   BB_upper  \
Ticker                                                                      
Date                                                                        
2019-03-14     1.084459      0.909991      2.605699  75.741614  43.461665   
2019-03-15     1.236875      1.056405      2.685605  76.985967  43.935529   
2019-03-18     1.189754      1.213716      2.658819  78.724377  44.445725   
2019-03-19     0.798858      1.284054      2.660448  73.527111  44.769369   
2019-03-20     0.701206      1.365200      2.659538  80.397000  45.127346   

Price        BB_lower return_lag1 return_lag5 return_lag10  
Ticker                                                      
Date                                                        
2019-03-14  39.821701    0.004422   -0.011575    -0.009836  
2019-03-15  39.709909    0.011116    0.002377     0.010511  
2019-03-18  39.590860    0.013008    0.034642     0.005029  
2019-03-19  39.63

In [7]:
#independent
features = ['SMA_7','SMA_21','SMA_50','RSI_14','volatility_21','volatility_50','return_lag1','return_lag5','return_lag10']
#dependent
target = ['Close']

x = df[features]
y = df[target]

#let's split the data based on time, no shuffling
split_idx = int(len(df) *0.8)

x_train, x_test = x.iloc[:split_idx], x.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {x_train.shape}, Test: {x_test.shape}')

Train: (967, 9), Test: (242, 9)


In [8]:
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [9]:
print('Phase 1 complete.')
print(f'Training rows: {len(x_train)}| Test rows: {len(x_test)}')
print(f'Features: {features}')

Phase 1 complete.
Training rows: 967| Test rows: 242
Features: ['SMA_7', 'SMA_21', 'SMA_50', 'RSI_14', 'volatility_21', 'volatility_50', 'return_lag1', 'return_lag5', 'return_lag10']


In [10]:
#save the data we just stored
import pickle

with open("../data/train_test_data.pk1","wb") as f:
    pickle.dump({
        "x_trained_scaled": x_train_scaled,
        "x_test_scaled": x_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "features" : features
    }, f)
print(f'Data saved.')

Data saved.
